In [ ]:
# ==================================================================================================
# PATIENT 24 A4D4D3D2 COMPOSITE WAVELET-COEFFICIENT VARIANCE TOPOMAP CLASSIFICATION
# MEDGEMMA-4B-IT + STANDARD LoRA
# ==================================================================================================
#
# BASE MODEL:
#     google/medgemma-4b-it
#
# METHOD:
#     STANDARD LoRA
#
# DATA:
#     Original training pool = 100 seizure + 400 non-seizure = 500
#     Actual training        =  80 seizure + 320 non-seizure = 400
#     Validation             =  20 seizure +  80 non-seizure = 100
#     Final testing          = 100 seizure + 400 non-seizure = 500
#
# LoRA:
#     rank    = 8
#     alpha   = 16
#     dropout = 0.05
#
# Training:
#     maximum epochs        = 50
#     early stopping        = patience 7
#     physical batch        = 1
#     gradient accumulation = 8
#     effective batch       = 8
#     learning rate         = 1e-4
#
# IMPORTANT:
#     - NO 4-bit quantization.
#     - NO 8-bit quantization.
#     - This is STANDARD LoRA, not QLoRA.
#     - BF16 is used when supported by the GPU.
#     - Final testing data are never used for optimization.
#     - Best epoch is selected by VALIDATION BALANCED ACCURACY.
#     - Validation loss is used only as a tie-breaker.
#     - System/user/image prompt tokens are masked from training loss.
#     - Only the assistant target "Seizure" or "Non-Seizure" is supervised.
#     - Language attention q/k/v/o projections are adapted.
#     - Vision encoder remains frozen.
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import os
import re
import gc
import sys
import json
import math
import time
import random
import warnings

import numpy as np
import pandas as pd

from PIL import Image

import torch

from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    f1_score,
    balanced_accuracy_score,
)

from huggingface_hub import whoami

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    get_linear_schedule_with_warmup,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
)

import transformers
import peft


# ==================================================================================================
# 1. CLEAN ANY MODEL LEFT FROM A PREVIOUS RUN
# ==================================================================================================

print("=" * 100)
print("CLEANING PREVIOUS GPU OBJECTS")
print("=" * 100)

for variable_name in [
    "model",
    "base_model",
    "base_model_for_test",
    "processor",
    "optimizer",
    "scheduler",
]:
    if variable_name in globals():
        try:
            del globals()[variable_name]
        except Exception:
            pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("\nPrevious GPU objects cleared.")


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

TRAINING_METADATA_CSV = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\Patiient 24\A4D4D3D2\Training"
    r"\Patient_24_A4D4D3D2_training_metadata.csv"
)

TESTING_METADATA_CSV = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\Patiient 24\A4D4D3D2\Testing\Blinded"
    r"\Patient_24_A4D4D3D2_blinded_test_labels.csv"
)

OUTPUT_ROOT = (
    r"C:\Users\ga_sd360\OneDrive - The University of Akron"
    r"\Desktop\Fine Tuning\Patiient 24\A4D4D3D2"
    r"\MedGemma4B_A4D4D3D2_LoRA_r8_a16_50epochs_patience7"
)

# OneDrive-safe checkpoint design.
BEST_ADAPTER_ROOT = os.path.join(
    OUTPUT_ROOT,
    "best_lora_checkpoints",
)

os.makedirs(
    BEST_ADAPTER_ROOT,
    exist_ok=True,
)

BEST_ADAPTER_DIR = None

TRAIN_SPLIT_CSV = os.path.join(
    OUTPUT_ROOT,
    "Patient_24_A4D4D3D2_LoRA_actual_train.csv",
)

VALIDATION_SPLIT_CSV = os.path.join(
    OUTPUT_ROOT,
    "Patient_24_A4D4D3D2_LoRA_validation.csv",
)

TRAINING_HISTORY_CSV = os.path.join(
    OUTPUT_ROOT,
    "Patient_24_A4D4D3D2_LoRA_training_history.csv",
)

FINAL_PREDICTIONS_CSV = os.path.join(
    OUTPUT_ROOT,
    "Patient_24_A4D4D3D2_LoRA_final_test_predictions.csv",
)

FINAL_METRICS_JSON = os.path.join(
    OUTPUT_ROOT,
    "Patient_24_A4D4D3D2_LoRA_final_metrics.json",
)

EXPERIMENT_CONFIG_JSON = os.path.join(
    OUTPUT_ROOT,
    "Patient_24_A4D4D3D2_LoRA_experiment_config.json",
)

TARGET_MODULES_TXT = os.path.join(
    OUTPUT_ROOT,
    "Patient_24_A4D4D3D2_LoRA_target_modules.txt",
)

CONFUSION_MATRIX_CSV = os.path.join(
    OUTPUT_ROOT,
    "Patient_24_A4D4D3D2_LoRA_confusion_matrix.csv",
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True,
)


# ==================================================================================================
# 3. MODEL
# ==================================================================================================

MODEL_ID = "google/medgemma-4b-it"


# ==================================================================================================
# 4. SYSTEM PROMPT
# ==================================================================================================

SYSTEM_PROMPT = """
You are a strict vision-language classifier for EEG seizure detection.
Classify each EEG topomap image into exactly one of two labels:
Seizure
Non-Seizure
Return only the final label. Do not provide explanation, punctuation, confidence score or extra text.
""".strip()


# ==================================================================================================
# 5. EXACT A4D4D3D2 USER PROMPT PROVIDED BY USER
# ==================================================================================================

USER_PROMPT = """
You are an expert in classifying seizure and non-seizure EEG patterns from composite wavelet-coefficient variance topomaps.
The attached composite image represents one 2-second EEG window.
In each composite image, the panels from left to right are: A4 approximation-coefficient variance (approximately 0–8 Hz), D4 detail-coefficient variance (approximately 8–16 Hz), D3 detail-coefficient variance (approximately 16–32 Hz), and D2 detail-coefficient variance (approximately 32–64 Hz).
The EEG was divided into non-overlapping 2-second windows, filtered using a fourth-order Butterworth 59–61 Hz band-stop filter, and decomposed using a level-4 Discrete Wavelet Transform. For each bipolar EEG channel, the A4, D4, D3, and D2 coefficient variances were calculated and spatially interpolated at the midpoint between the two electrodes to create the topomaps.
Each panel uses its own fixed coefficient-specific color scale across the complete dataset. Therefore, evaluate the relative intensity and spatial distribution within each panel independently. Do not assume that the same color represents the same numerical variance across different panels. Yellow indicates relatively higher coefficient variance within the corresponding panel, while dark purple indicates relatively lower coefficient variance.
Internally evaluate the image in five steps:
Examine the A4 panel for the intensity, spatial extent, localization, symmetry, and organization of elevated variance.
Examine the D4 panel independently using the same characteristics.
Examine the D3 panel independently using the same characteristics.
Examine the D2 panel independently using the same characteristics.
Combine the evidence from all four panels and determine whether the overall composite pattern is more consistent with seizure or non-seizure activity.
Seizure-related evidence may appear across all four panels or may be more prominent in only a subset of panels. Do not require all panels to show equally strong elevated variance before predicting seizure.
Consider whether elevated variance is focal or widespread, symmetric or asymmetric, and spatially organized or irregular. Evaluate all four panels together rather than relying on brightness, one isolated region, or any single panel as a definitive feature.
Return exactly one label:
Seizure
Non-Seizure
""".strip()


# ==================================================================================================
# 6. RANDOM SEED
# ==================================================================================================

RANDOM_SEED = 42


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


set_all_seeds(RANDOM_SEED)


# ==================================================================================================
# 7. LoRA HYPERPARAMETERS
# ==================================================================================================

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

LORA_TARGET_LEAF_NAMES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
]


# ==================================================================================================
# 8. TRAINING HYPERPARAMETERS
# ==================================================================================================

NUM_EPOCHS = 50

EARLY_STOPPING_PATIENCE = 7

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01

TRAIN_BATCH_SIZE = 1
VALIDATION_BATCH_SIZE = 1

GRADIENT_ACCUMULATION_STEPS = 8

MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.05

MAX_NEW_TOKENS = 8

NUM_WORKERS = 0


# ==================================================================================================
# 9. EXPECTED COUNTS
# ==================================================================================================

EXPECTED_FULL_TRAIN_SEIZURE = 100
EXPECTED_FULL_TRAIN_NON_SEIZURE = 400

EXPECTED_ACTUAL_TRAIN_SEIZURE = 80
EXPECTED_ACTUAL_TRAIN_NON_SEIZURE = 320

EXPECTED_VALIDATION_SEIZURE = 20
EXPECTED_VALIDATION_NON_SEIZURE = 80

EXPECTED_TEST_SEIZURE = 100
EXPECTED_TEST_NON_SEIZURE = 400


# ==================================================================================================
# 10. HUGGING FACE LOGIN CHECK
# ==================================================================================================

print("\n" + "=" * 100)
print("HUGGING FACE LOGIN CHECK")
print("=" * 100)

try:
    hf_info = whoami()

    print(
        "\nLogged in as:",
        hf_info["name"],
    )

except Exception as error:
    raise RuntimeError(
        "\nHugging Face authentication is not available.\n"
        "Run huggingface_hub.login() first."
    ) from error


# ==================================================================================================
# 11. GPU CHECK
# ==================================================================================================

print("\n" + "=" * 100)
print("GPU CHECK")
print("=" * 100)

if not torch.cuda.is_available():
    raise RuntimeError(
        "\nCUDA GPU was not detected."
    )

DEVICE = torch.device("cuda:0")

GPU_NAME = torch.cuda.get_device_name(0)

GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory
    /
    1024 ** 3
)

print(f"\nGPU: {GPU_NAME}")
print(f"GPU memory: {GPU_MEMORY_GB:.2f} GB")


# ==================================================================================================
# 12. PRECISION
# ==================================================================================================

if torch.cuda.is_bf16_supported():
    MODEL_DTYPE = torch.bfloat16
    PRECISION_NAME = "bfloat16"

else:
    MODEL_DTYPE = torch.float16
    PRECISION_NAME = "float16"

print(
    f"Training/model dtype: {PRECISION_NAME}"
)

torch.backends.cuda.matmul.allow_tf32 = True


# ==================================================================================================
# 13. PACKAGE VERSIONS
# ==================================================================================================

print("\n" + "=" * 100)
print("PACKAGE VERSIONS")
print("=" * 100)

print(f"\nPython       : {sys.version.split()[0]}")
print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"PEFT         : {peft.__version__}")


# ==================================================================================================
# 14. CLEAN PATH
# ==================================================================================================

def clean_path(value):

    if pd.isna(value):
        return ""

    value = str(value).strip()
    value = value.strip('"')
    value = value.strip("'")
    value = os.path.expandvars(value)
    value = os.path.expanduser(value)
    value = os.path.normpath(value)

    return value


# ==================================================================================================
# 15. DETECT IMAGE PATH COLUMN
# ==================================================================================================

def detect_image_path_column(df, dataset_type):

    if dataset_type == "test":

        candidates = [
            "blinded_image_path",
            "blinded_topomap_path",
            "blinded_path",
            "fine_tuning_image_path",
            "composite_image_path",
            "image_path",
            "selected_image_path",
            "original_image_path",
            "file_path",
            "filepath",
            "path",
        ]

    else:

        candidates = [
            "fine_tuning_image_path",
            "composite_image_path",
            "image_path",
            "selected_image_path",
            "blinded_image_path",
            "blinded_topomap_path",
            "blinded_path",
            "original_image_path",
            "file_path",
            "filepath",
            "path",
        ]

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    for column in df.columns:

        lower = str(column).lower()

        if (
            "path" in lower
            and
            (
                "image" in lower
                or
                "topomap" in lower
                or
                "blinded" in lower
            )
        ):
            return column

    raise ValueError(
        "\nCould not detect an image-path column.\n\n"
        f"Columns:\n{list(df.columns)}"
    )


# ==================================================================================================
# 16. DETECT LABEL COLUMN
# ==================================================================================================

def detect_label_column(df):

    candidates = [
        "standard_label",
        "label",
        "Label",
        "target",
        "class_label",
        "fine_tuning_class",
        "class_name",
        "class",
        "category",
        "true_label",
        "ground_truth",
    ]

    for candidate in candidates:

        if candidate in df.columns:
            return candidate

    for column in df.columns:

        lower = str(column).lower()

        if (
            "label" in lower
            or
            lower == "class"
        ):
            return column

    raise ValueError(
        "\nCould not detect a label column.\n\n"
        f"Columns:\n{list(df.columns)}"
    )


# ==================================================================================================
# 17. LABEL NORMALIZATION
#
# 1 = Seizure
# 0 = Non-Seizure
# ==================================================================================================

def normalize_ground_truth_label(value):

    if pd.isna(value):
        return None

    if isinstance(
        value,
        (
            int,
            np.integer,
        ),
    ):

        if int(value) == 1:
            return 1

        if int(value) == 0:
            return 0

    if isinstance(
        value,
        (
            float,
            np.floating,
        ),
    ):

        if float(value) == 1.0:
            return 1

        if float(value) == 0.0:
            return 0

    text = str(value).strip().lower()

    text = text.replace("_", " ")
    text = text.replace("–", "-")

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    # IMPORTANT: NON-SEIZURE FIRST.
    if text in {
        "0",
        "0.0",
        "non seizure",
        "non-seizure",
        "nonseizure",
        "normal",
        "interictal",
        "ns",
    }:
        return 0

    if (
        "non" in text
        and
        "seizure" in text
    ):
        return 0

    if text in {
        "1",
        "1.0",
        "seizure",
        "ictal",
        "sz",
    }:
        return 1

    if "seizure" in text:
        return 1

    return None


# ==================================================================================================
# 18. LOAD + VALIDATE METADATA
# ==================================================================================================

def load_and_standardize_metadata(
    csv_path,
    dataset_name,
    dataset_type,
):

    print("\n" + "=" * 100)

    print(
        f"LOADING {dataset_name.upper()}"
    )

    print("=" * 100)

    if not os.path.isfile(csv_path):

        raise FileNotFoundError(
            f"\nCSV does not exist:\n{csv_path}"
        )

    df = pd.read_csv(csv_path)

    print(
        f"\nCSV:\n{csv_path}"
    )

    print(
        f"\nRows: {len(df)}"
    )

    print("\nColumns:")

    for column in df.columns:

        print(
            f"  - {column}"
        )

    image_column = detect_image_path_column(
        df,
        dataset_type,
    )

    label_column = detect_label_column(
        df
    )

    print(
        f"\nDetected image path column : {image_column}"
    )

    print(
        f"Detected label column      : {label_column}"
    )

    df["_image_path"] = df[
        image_column
    ].apply(
        clean_path
    )

    df["_label"] = df[
        label_column
    ].apply(
        normalize_ground_truth_label
    )

    # ----------------------------------------------------------------------------------------------
    # LABEL CHECK
    # ----------------------------------------------------------------------------------------------

    unknown = df[
        df["_label"].isna()
    ]

    if len(unknown) > 0:

        print(
            "\nUnknown label values:"
        )

        print(
            unknown[
                [label_column]
            ]
            .drop_duplicates()
            .to_string(
                index=False
            )
        )

        raise ValueError(
            "\nSome labels could not be interpreted."
        )

    df["_label"] = df[
        "_label"
    ].astype(int)

    # ----------------------------------------------------------------------------------------------
    # IMAGE CHECK
    # ----------------------------------------------------------------------------------------------

    missing = []
    empty = []

    for image_path in df["_image_path"]:

        if not os.path.isfile(image_path):

            missing.append(
                image_path
            )

            continue

        try:

            if os.path.getsize(image_path) <= 0:

                empty.append(
                    image_path
                )

        except OSError:
            pass

    print(
        f"\nChecked: {len(df)} images"
    )

    print(
        f"Missing: {len(missing)}"
    )

    print(
        f"Empty  : {len(empty)}"
    )

    if missing:

        print(
            "\nFirst missing paths:"
        )

        for path in missing[:20]:
            print(path)

        raise FileNotFoundError(
            "\nSome image files are missing."
        )

    if empty:

        raise RuntimeError(
            "\nSome image files are empty."
        )

    duplicate_count = int(
        df["_image_path"].duplicated().sum()
    )

    if duplicate_count > 0:

        raise ValueError(
            f"\n{duplicate_count} duplicate image paths were found."
        )

    seizure_count = int(
        (
            df["_label"] == 1
        ).sum()
    )

    non_seizure_count = int(
        (
            df["_label"] == 0
        ).sum()
    )

    print("\nClass counts:")

    print(
        f"  Seizure     : {seizure_count}"
    )

    print(
        f"  Non-seizure : {non_seizure_count}"
    )

    print(
        f"  Total        : {len(df)}"
    )

    return (
        df,
        image_column,
        label_column,
    )


# ==================================================================================================
# 19. LOAD TRAINING POOL
# ==================================================================================================

(
    full_training_df,
    training_image_column,
    training_label_column,
) = load_and_standardize_metadata(
    TRAINING_METADATA_CSV,
    "Patient 24 A4D4D3D2 training pool",
    "train",
)


# ==================================================================================================
# 20. VERIFY TRAINING COUNTS
# ==================================================================================================

full_training_seizures = int(
    (
        full_training_df["_label"] == 1
    ).sum()
)

full_training_non_seizures = int(
    (
        full_training_df["_label"] == 0
    ).sum()
)

assert (
    full_training_seizures
    ==
    EXPECTED_FULL_TRAIN_SEIZURE
), (
    f"Expected {EXPECTED_FULL_TRAIN_SEIZURE} seizure training images, "
    f"found {full_training_seizures}"
)

assert (
    full_training_non_seizures
    ==
    EXPECTED_FULL_TRAIN_NON_SEIZURE
), (
    f"Expected {EXPECTED_FULL_TRAIN_NON_SEIZURE} non-seizure training images, "
    f"found {full_training_non_seizures}"
)


# ==================================================================================================
# 21. CREATE FIXED STRATIFIED TRAIN / VALIDATION SPLIT
# ==================================================================================================

seizure_pool = full_training_df[
    full_training_df["_label"] == 1
].copy()

non_seizure_pool = full_training_df[
    full_training_df["_label"] == 0
].copy()


seizure_pool = seizure_pool.sample(
    frac=1,
    random_state=RANDOM_SEED,
).reset_index(
    drop=True
)


non_seizure_pool = non_seizure_pool.sample(
    frac=1,
    random_state=RANDOM_SEED,
).reset_index(
    drop=True
)


# ----------------------------------------------------------------------------------------------
# ACTUAL TRAINING
# ----------------------------------------------------------------------------------------------

actual_train_seizure = seizure_pool.iloc[
    :EXPECTED_ACTUAL_TRAIN_SEIZURE
].copy()


actual_train_non_seizure = non_seizure_pool.iloc[
    :EXPECTED_ACTUAL_TRAIN_NON_SEIZURE
].copy()


# ----------------------------------------------------------------------------------------------
# VALIDATION
# ----------------------------------------------------------------------------------------------

validation_seizure = seizure_pool.iloc[
    EXPECTED_ACTUAL_TRAIN_SEIZURE:
    EXPECTED_ACTUAL_TRAIN_SEIZURE
    +
    EXPECTED_VALIDATION_SEIZURE
].copy()


validation_non_seizure = non_seizure_pool.iloc[
    EXPECTED_ACTUAL_TRAIN_NON_SEIZURE:
    EXPECTED_ACTUAL_TRAIN_NON_SEIZURE
    +
    EXPECTED_VALIDATION_NON_SEIZURE
].copy()


actual_train_df = pd.concat(
    [
        actual_train_seizure,
        actual_train_non_seizure,
    ],
    ignore_index=True,
)


validation_df = pd.concat(
    [
        validation_seizure,
        validation_non_seizure,
    ],
    ignore_index=True,
)


actual_train_df = actual_train_df.sample(
    frac=1,
    random_state=RANDOM_SEED,
).reset_index(
    drop=True
)


validation_df = validation_df.sample(
    frac=1,
    random_state=RANDOM_SEED,
).reset_index(
    drop=True
)


# ==================================================================================================
# 22. VERIFY INTERNAL SPLIT
# ==================================================================================================

actual_train_seizure_count = int(
    (
        actual_train_df["_label"] == 1
    ).sum()
)

actual_train_non_seizure_count = int(
    (
        actual_train_df["_label"] == 0
    ).sum()
)

validation_seizure_count = int(
    (
        validation_df["_label"] == 1
    ).sum()
)

validation_non_seizure_count = int(
    (
        validation_df["_label"] == 0
    ).sum()
)


assert (
    actual_train_seizure_count
    ==
    EXPECTED_ACTUAL_TRAIN_SEIZURE
)

assert (
    actual_train_non_seizure_count
    ==
    EXPECTED_ACTUAL_TRAIN_NON_SEIZURE
)

assert (
    validation_seizure_count
    ==
    EXPECTED_VALIDATION_SEIZURE
)

assert (
    validation_non_seizure_count
    ==
    EXPECTED_VALIDATION_NON_SEIZURE
)


train_validation_overlap = set(
    actual_train_df["_image_path"]
).intersection(
    set(
        validation_df["_image_path"]
    )
)

assert len(
    train_validation_overlap
) == 0


print("\n" + "=" * 100)
print("TRAIN / VALIDATION SPLIT")
print("=" * 100)


print("\nACTUAL TRAINING")

print(
    f"Seizure     : {actual_train_seizure_count}"
)

print(
    f"Non-seizure : {actual_train_non_seizure_count}"
)

print(
    f"Total        : {len(actual_train_df)}"
)


print("\nVALIDATION")

print(
    f"Seizure     : {validation_seizure_count}"
)

print(
    f"Non-seizure : {validation_non_seizure_count}"
)

print(
    f"Total        : {len(validation_df)}"
)

print(
    "\nTrain/validation image overlap: 0"
)


# ==================================================================================================
# 23. SAVE EXACT TRAIN / VALIDATION SPLITS
# ==================================================================================================

actual_train_df.to_csv(
    TRAIN_SPLIT_CSV,
    index=False,
)

validation_df.to_csv(
    VALIDATION_SPLIT_CSV,
    index=False,
)


print(
    "\nTraining split saved:\n"
    f"{TRAIN_SPLIT_CSV}"
)

print(
    "\nValidation split saved:\n"
    f"{VALIDATION_SPLIT_CSV}"
)


# ==================================================================================================
# 24. LOAD FINAL BLINDED TESTING METADATA
# ==================================================================================================

(
    testing_df,
    testing_image_column,
    testing_label_column,
) = load_and_standardize_metadata(
    TESTING_METADATA_CSV,
    "Patient 24 A4D4D3D2 final blinded testing",
    "test",
)


# ==================================================================================================
# 25. VERIFY TEST COUNTS
# ==================================================================================================

test_seizure_count = int(
    (
        testing_df["_label"] == 1
    ).sum()
)

test_non_seizure_count = int(
    (
        testing_df["_label"] == 0
    ).sum()
)


assert (
    test_seizure_count
    ==
    EXPECTED_TEST_SEIZURE
), (
    f"Expected {EXPECTED_TEST_SEIZURE} test seizures, "
    f"found {test_seizure_count}"
)


assert (
    test_non_seizure_count
    ==
    EXPECTED_TEST_NON_SEIZURE
), (
    f"Expected {EXPECTED_TEST_NON_SEIZURE} test non-seizures, "
    f"found {test_non_seizure_count}"
)


# ==================================================================================================
# 26. STRONG TRAIN / TEST LEAKAGE CHECK
# ==================================================================================================

def normalize_identity_value(value):

    if pd.isna(value):
        return None

    text = str(value).strip()

    try:

        number = float(text)

        if number.is_integer():

            return str(
                int(number)
            )

    except Exception:
        pass

    return text.lower()


def check_train_test_identity_leakage(
    train_df,
    test_df,
):

    serial_aliases = [
        "global_serial_id",
        "global_serial",
        "Global_Serial_ID",
        "Global_Serial",
        "Serial",
        "serial",
    ]

    train_serial_column = next(
        (
            column
            for column in serial_aliases
            if column in train_df.columns
        ),
        None,
    )

    test_serial_column = next(
        (
            column
            for column in serial_aliases
            if column in test_df.columns
        ),
        None,
    )

    if (
        train_serial_column is not None
        and
        test_serial_column is not None
    ):

        train_ids = {

            normalize_identity_value(value)

            for value
            in train_df[train_serial_column]

            if normalize_identity_value(value) is not None
        }

        test_ids = {

            normalize_identity_value(value)

            for value
            in test_df[test_serial_column]

            if normalize_identity_value(value) is not None
        }

        overlap = train_ids.intersection(
            test_ids
        )

        identity_description = (
            f"{train_serial_column} <-> {test_serial_column}"
        )

        print(
            f"\nLeakage identity columns used: {identity_description}"
        )

        print(
            f"Identity overlap count         : {len(overlap)}"
        )

        if overlap:

            raise RuntimeError(
                "\nDATA LEAKAGE DETECTED using global serial identity.\n"
                f"{len(overlap)} original EEG windows occur in both "
                "training and final blinded testing."
            )

        return identity_description


    fallback_identity_columns = [
        "original_image_path",
        "original_image_name",
        "original_filename",
    ]


    for column in fallback_identity_columns:

        if (
            column in train_df.columns
            and
            column in test_df.columns
        ):

            train_ids = {

                normalize_identity_value(value)

                for value
                in train_df[column]

                if normalize_identity_value(value) is not None
            }

            test_ids = {

                normalize_identity_value(value)

                for value
                in test_df[column]

                if normalize_identity_value(value) is not None
            }

            overlap = train_ids.intersection(
                test_ids
            )

            print(
                f"\nLeakage identity column used: {column}"
            )

            print(
                f"Identity overlap count       : {len(overlap)}"
            )

            if overlap:

                raise RuntimeError(
                    f"\nDATA LEAKAGE DETECTED using column '{column}'.\n"
                    f"{len(overlap)} original samples occur in both "
                    "training and final blinded testing."
                )

            return column


    warnings.warn(
        "\nNo global serial/original-image identity columns "
        "were available for a strong train/test leakage check."
    )

    return None


LEAKAGE_ID_COLUMN = check_train_test_identity_leakage(
    full_training_df,
    testing_df,
)


current_path_overlap = set(
    full_training_df["_image_path"]
).intersection(
    set(
        testing_df["_image_path"]
    )
)


if current_path_overlap:

    raise RuntimeError(
        "\nIdentical current image paths occur "
        "in both training and final blinded testing."
    )


print(
    "\nCurrent image-path overlap: 0"
)


# ==================================================================================================
# 27. DATASET CLASS
# ==================================================================================================

class EEGTopomapDataset(Dataset):

    def __init__(
        self,
        dataframe,
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

    def __len__(self):

        return len(
            self.df
        )

    def __getitem__(
        self,
        index,
    ):

        row = self.df.iloc[
            index
        ]

        return {

            "index": int(index),

            "image_path": str(
                row["_image_path"]
            ),

            "label": int(
                row["_label"]
            ),
        }


# ==================================================================================================
# 28. LOAD MEDGEMMA PROCESSOR
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING MEDGEMMA PROCESSOR")
print("=" * 100)


processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=True,
)


if hasattr(
    processor,
    "tokenizer",
):

    processor.tokenizer.padding_side = "right"


print(
    "\nProcessor loaded successfully."
)


# ==================================================================================================
# 29. MEDGEMMA MODEL LOADING FUNCTION
# ==================================================================================================

def load_medgemma_base():

    print(
        f"\nLoading {MODEL_ID}..."
    )

    try:

        loaded_model = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID,
            token=True,
            dtype=MODEL_DTYPE,
            low_cpu_mem_usage=True,
            attn_implementation="sdpa",
        )

    except TypeError:

        loaded_model = AutoModelForMultimodalLM.from_pretrained(
            MODEL_ID,
            token=True,
            torch_dtype=MODEL_DTYPE,
            low_cpu_mem_usage=True,
            attn_implementation="sdpa",
        )

    loaded_model = loaded_model.to(
        DEVICE
    )

    return loaded_model


# ==================================================================================================
# 30. LOAD BASE MEDGEMMA
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING MEDGEMMA-4B BASE MODEL")
print("=" * 100)


gc.collect()
torch.cuda.empty_cache()


base_model = load_medgemma_base()


print(
    "\nSUCCESS: MedGemma-4B loaded on GPU."
)

print(
    f"GPU allocated after base-model load: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print(
    f"GPU reserved after base-model load : "
    f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
)


# ==================================================================================================
# 31. TRAINING MEMORY SETTINGS
# ==================================================================================================

if hasattr(
    base_model.config,
    "use_cache",
):

    base_model.config.use_cache = False


try:

    base_model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={
            "use_reentrant": False
        }
    )

except TypeError:

    base_model.gradient_checkpointing_enable()


print(
    "\nGradient checkpointing enabled."
)


# ==================================================================================================
# 32. FIND LoRA TARGET MODULES
#
# LANGUAGE ATTENTION ONLY
# VISION ENCODER REMAINS FROZEN
# ==================================================================================================

print("\n" + "=" * 100)
print("SEARCHING FOR LoRA TARGET MODULES")
print("=" * 100)


possible_attention_modules = []


for module_name, module in base_model.named_modules():

    leaf_name = module_name.split(
        "."
    )[-1]

    if (
        leaf_name in LORA_TARGET_LEAF_NAMES
        and
        isinstance(
            module,
            torch.nn.Linear,
        )
    ):

        possible_attention_modules.append(
            module_name
        )


print(
    f"\nTotal q/k/v/o Linear modules found: "
    f"{len(possible_attention_modules)}"
)


language_attention_modules = [

    name

    for name
    in possible_attention_modules

    if "language_model" in name.lower()
]


if language_attention_modules:

    LORA_TARGET_MODULES = language_attention_modules

    print(
        "\nUsing language-model q/k/v/o attention projections."
    )


else:

    LORA_TARGET_MODULES = [

        name

        for name
        in possible_attention_modules

        if (
            "vision" not in name.lower()
            and
            "image_encoder" not in name.lower()
        )
    ]

    warnings.warn(
        "\nNo module names contained 'language_model'. "
        "Using q/k/v/o modules after excluding vision modules."
    )


if not LORA_TARGET_MODULES:

    raise RuntimeError(
        "\nNo suitable language attention modules were found for LoRA."
    )


print(
    f"\nActual LoRA target module count: "
    f"{len(LORA_TARGET_MODULES)}"
)


print(
    "\nFirst 20 target modules:"
)


for name in LORA_TARGET_MODULES[:20]:

    print(
        "  ",
        name,
    )


if len(LORA_TARGET_MODULES) > 20:

    print(
        f"  ... plus "
        f"{len(LORA_TARGET_MODULES) - 20} more."
    )


with open(
    TARGET_MODULES_TXT,
    "w",
    encoding="utf-8",
) as file:

    for name in LORA_TARGET_MODULES:

        file.write(
            name
            +
            "\n"
        )


# ==================================================================================================
# 33. LoRA CONFIG
# ==================================================================================================

lora_config = LoraConfig(

    r=LORA_R,

    lora_alpha=LORA_ALPHA,

    lora_dropout=LORA_DROPOUT,

    target_modules=LORA_TARGET_MODULES,

    bias="none",

    inference_mode=False,
)


# ==================================================================================================
# 34. INJECT LoRA
# ==================================================================================================

print("\n" + "=" * 100)
print("INJECTING LoRA")
print("=" * 100)


model = get_peft_model(
    base_model,
    lora_config,
)


if hasattr(
    model,
    "enable_input_require_grads",
):

    try:

        model.enable_input_require_grads()

    except Exception:
        pass


# ==================================================================================================
# 35. TRAINABLE PARAMETER REPORT
# ==================================================================================================

print("\n" + "=" * 100)
print("TRAINABLE PARAMETER REPORT")
print("=" * 100)


model.print_trainable_parameters()


trainable_parameter_count = sum(

    parameter.numel()

    for parameter
    in model.parameters()

    if parameter.requires_grad
)


total_parameter_count = sum(

    parameter.numel()

    for parameter
    in model.parameters()
)


trainable_percentage = (

    100.0

    *

    trainable_parameter_count

    /

    total_parameter_count
)


print(
    f"\nTrainable parameters : {trainable_parameter_count:,}"
)

print(
    f"Total parameters     : {total_parameter_count:,}"
)

print(
    f"Trainable percentage : {trainable_percentage:.6f}%"
)


# ==================================================================================================
# 36. VERIFY VISION PARAMETERS ARE NOT TRAINABLE
# ==================================================================================================

vision_trainable_parameters = [

    name

    for name, parameter
    in model.named_parameters()

    if (
        parameter.requires_grad
        and
        "vision" in name.lower()
    )
]


print(
    f"\nTrainable parameters containing 'vision': "
    f"{len(vision_trainable_parameters)}"
)


if vision_trainable_parameters:

    print(
        "\nWARNING: Some vision parameters appear trainable."
    )

    for name in vision_trainable_parameters[:20]:

        print(
            "  ",
            name,
        )


# ==================================================================================================
# 37. TARGET TEXT
# ==================================================================================================

def label_to_target_text(
    label,
):

    if int(label) == 1:

        return "Seizure"

    return "Non-Seizure"


# ==================================================================================================
# 38. BUILD PROMPT
# ==================================================================================================

def build_prompt_messages(
    image,
):

    return [

        {
            "role": "system",

            "content": [

                {
                    "type": "text",
                    "text": SYSTEM_PROMPT,
                }

            ],
        },

        {
            "role": "user",

            "content": [

                {
                    "type": "text",
                    "text": USER_PROMPT,
                },

                {
                    "type": "image",
                    "image": image,
                },

            ],
        },

    ]


# ==================================================================================================
# 39. BUILD FULL TRAINING CONVERSATION
# ==================================================================================================

def build_training_messages(
    image,
    target_text,
):

    messages = build_prompt_messages(
        image
    )

    messages.append(

        {
            "role": "assistant",

            "content": [

                {
                    "type": "text",
                    "text": target_text,
                }

            ],
        }

    )

    return messages


# ==================================================================================================
# 40. TRAINING COLLATOR
#
# Physical batch size = 1
#
# SYSTEM + USER + IMAGE + ASSISTANT HEADER -> masked
# Only Seizure / Non-Seizure target -> supervised
# ==================================================================================================

class MedGemmaTrainingCollator:

    def __init__(
        self,
        processor,
    ):

        self.processor = processor


    def __call__(
        self,
        examples,
    ):

        if len(examples) != 1:

            raise ValueError(
                "\nThis pipeline expects TRAIN_BATCH_SIZE = 1."
            )


        example = examples[0]


        image_path = example[
            "image_path"
        ]


        label = int(
            example[
                "label"
            ]
        )


        target_text = label_to_target_text(
            label
        )


        with Image.open(
            image_path
        ) as opened_image:


            image = opened_image.convert(
                "RGB"
            )


            # --------------------------------------------------------------------------------------
            # PROMPT ONLY
            # --------------------------------------------------------------------------------------

            prompt_messages = build_prompt_messages(
                image
            )


            prompt_inputs = self.processor.apply_chat_template(

                prompt_messages,

                add_generation_prompt=True,

                tokenize=True,

                return_dict=True,

                return_tensors="pt",
            )


            prompt_length = int(
                prompt_inputs[
                    "input_ids"
                ].shape[
                    1
                ]
            )


            # --------------------------------------------------------------------------------------
            # FULL SUPERVISED CONVERSATION
            # --------------------------------------------------------------------------------------

            full_messages = build_training_messages(
                image,
                target_text,
            )


            full_inputs = self.processor.apply_chat_template(

                full_messages,

                add_generation_prompt=False,

                tokenize=True,

                return_dict=True,

                return_tensors="pt",
            )


        full_ids = full_inputs[
            "input_ids"
        ]


        prompt_ids = prompt_inputs[
            "input_ids"
        ]


        if full_ids.shape[1] <= prompt_length:

            raise RuntimeError(
                "\nFull sequence contains no assistant target tokens."
            )


        if not torch.equal(

            full_ids[
                :,
                :prompt_length
            ],

            prompt_ids,

        ):

            raise RuntimeError(
                "\nMEDGEMMA CHAT-TEMPLATE PREFIX MISMATCH.\n\n"
                "The code stopped rather than train with "
                "an incorrect assistant-only loss mask."
            )


        labels = full_ids.clone()


        labels[
            :,
            :prompt_length
        ] = -100


        trainable_token_count = int(
            (
                labels != -100
            ).sum()
        )


        if trainable_token_count <= 0:

            raise RuntimeError(
                "\nNo assistant target tokens remain after masking."
            )


        full_inputs[
            "labels"
        ] = labels


        del prompt_inputs


        return full_inputs


# ==================================================================================================
# 41. DATASETS
# ==================================================================================================

train_dataset = EEGTopomapDataset(
    actual_train_df
)


validation_dataset = EEGTopomapDataset(
    validation_df
)


training_collator = MedGemmaTrainingCollator(
    processor
)


# ==================================================================================================
# 42. DATA LOADERS
# ==================================================================================================

train_generator = torch.Generator()


train_generator.manual_seed(
    RANDOM_SEED
)


train_loader = DataLoader(

    train_dataset,

    batch_size=TRAIN_BATCH_SIZE,

    shuffle=True,

    generator=train_generator,

    num_workers=NUM_WORKERS,

    collate_fn=training_collator,

    pin_memory=False,
)


validation_loss_loader = DataLoader(

    validation_dataset,

    batch_size=VALIDATION_BATCH_SIZE,

    shuffle=False,

    num_workers=NUM_WORKERS,

    collate_fn=training_collator,

    pin_memory=False,
)


# ==================================================================================================
# 43. MOVE BATCH TO GPU
# ==================================================================================================

def move_batch_to_device(
    batch,
):

    moved = {}


    for key, value in batch.items():


        if not torch.is_tensor(value):

            moved[key] = value

            continue


        if value.dtype.is_floating_point:

            moved[key] = value.to(
                device=DEVICE,
                dtype=MODEL_DTYPE,
            )


        else:

            moved[key] = value.to(
                DEVICE
            )


    return moved


# ==================================================================================================
# 44. MIXED PRECISION CONTEXT
# ==================================================================================================

def autocast_context():

    return torch.autocast(
        device_type="cuda",
        dtype=MODEL_DTYPE,
    )


# ==================================================================================================
# 45. PRE-FLIGHT COLLATOR + FORWARD TEST
# ==================================================================================================

print("\n" + "=" * 100)
print("PRE-FLIGHT TRAINING EXAMPLE TEST")
print("=" * 100)


preview_example = train_dataset[
    0
]


print(
    "\nImage:"
)


print(
    preview_example[
        "image_path"
    ]
)


print(
    "\nGround-truth target:"
)


print(
    label_to_target_text(
        preview_example[
            "label"
        ]
    )
)


preview_batch_cpu = training_collator(
    [
        preview_example
    ]
)


preview_target_ids = preview_batch_cpu[
    "labels"
][0]


preview_target_ids = preview_target_ids[
    preview_target_ids != -100
]


print(
    "\nNumber of assistant target tokens:",
    len(
        preview_target_ids
    ),
)


try:

    decoded_training_target = processor.decode(

        preview_target_ids,

        skip_special_tokens=False,
    )


    print(
        "\nDecoded supervised target:"
    )


    print(
        repr(
            decoded_training_target
        )
    )


except Exception:
    pass


preview_batch_gpu = move_batch_to_device(
    preview_batch_cpu
)


model.eval()


with torch.no_grad():

    with autocast_context():

        preview_outputs = model(
            **preview_batch_gpu
        )


preview_loss = float(

    preview_outputs.loss
    .detach()
    .float()
    .cpu()
)


print(
    f"\nPre-flight forward loss: {preview_loss:.6f}"
)


if not np.isfinite(
    preview_loss
):

    raise RuntimeError(
        "\nPre-flight loss is NaN or infinite."
    )


print(
    "\nSUCCESS: MedGemma + LoRA training forward pass works."
)


del preview_batch_cpu
del preview_batch_gpu
del preview_outputs


gc.collect()
torch.cuda.empty_cache()


# ==================================================================================================
# 46. OPTIMIZER
# ==================================================================================================

trainable_parameters = [

    parameter

    for parameter
    in model.parameters()

    if parameter.requires_grad
]


optimizer = torch.optim.AdamW(

    trainable_parameters,

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY,
)


# ==================================================================================================
# 47. SCHEDULER
# ==================================================================================================

optimizer_steps_per_epoch = math.ceil(

    len(
        train_loader
    )

    /

    GRADIENT_ACCUMULATION_STEPS
)


total_optimizer_steps = (

    optimizer_steps_per_epoch

    *

    NUM_EPOCHS
)


warmup_steps = int(

    total_optimizer_steps

    *

    WARMUP_RATIO
)


scheduler = get_linear_schedule_with_warmup(

    optimizer,

    num_warmup_steps=warmup_steps,

    num_training_steps=total_optimizer_steps,
)


print("\n" + "=" * 100)
print("TRAINING SETTINGS")
print("=" * 100)


print(
    f"\nEpochs                    : {NUM_EPOCHS}"
)

print(
    f"Early stopping patience   : {EARLY_STOPPING_PATIENCE}"
)

print(
    f"Training images           : {len(train_dataset)}"
)

print(
    f"Validation images         : {len(validation_dataset)}"
)

print(
    f"Physical batch size       : {TRAIN_BATCH_SIZE}"
)

print(
    f"Gradient accumulation     : {GRADIENT_ACCUMULATION_STEPS}"
)

print(
    f"Effective batch size      : "
    f"{TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}"
)

print(
    f"Optimizer steps/epoch     : {optimizer_steps_per_epoch}"
)

print(
    f"Total optimizer steps     : {total_optimizer_steps}"
)

print(
    f"Warmup steps              : {warmup_steps}"
)

print(
    f"Learning rate             : {LEARNING_RATE}"
)

print(
    f"LoRA rank                 : {LORA_R}"
)

print(
    f"LoRA alpha                : {LORA_ALPHA}"
)

print(
    f"LoRA dropout              : {LORA_DROPOUT}"
)


# ==================================================================================================
# 48. FP16 GRAD SCALER
# ==================================================================================================

if MODEL_DTYPE == torch.float16:

    try:

        grad_scaler = torch.amp.GradScaler(
            "cuda"
        )

    except Exception:

        grad_scaler = torch.cuda.amp.GradScaler()


else:

    grad_scaler = None


# ==================================================================================================
# 49. VALIDATION LOSS
# ==================================================================================================

@torch.no_grad()
def calculate_validation_loss():

    model.eval()

    losses = []


    progress = tqdm(

        validation_loss_loader,

        desc="Validation loss",

        leave=False,
    )


    for batch in progress:


        batch = move_batch_to_device(
            batch
        )


        with autocast_context():


            outputs = model(
                **batch
            )


            loss = outputs.loss


        loss_value = float(

            loss
            .detach()
            .float()
            .cpu()
        )


        losses.append(
            loss_value
        )


        progress.set_postfix(
            loss=f"{loss_value:.4f}"
        )


        del batch
        del outputs
        del loss


    if not losses:

        return float(
            "nan"
        )


    return float(
        np.mean(
            losses
        )
    )


# ==================================================================================================
# 50. NORMALIZE MODEL PREDICTION
# ==================================================================================================

def normalize_prediction_text(
    output_text,
):

    text = str(
        output_text
    ).strip().lower()


    text = text.replace(
        "_",
        " ",
    )


    text = text.replace(
        "–",
        "-",
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    # IMPORTANT:
    # Check Non-Seizure before Seizure.
    if re.search(
        r"\bnon[\s-]*seizure\b",
        text,
    ):

        return 0


    if re.search(
        r"\bseizure\b",
        text,
    ):

        return 1


    return None


# ==================================================================================================
# 51. SINGLE IMAGE INFERENCE
# ==================================================================================================

@torch.no_grad()
def predict_single_image(
    image_path,
):

    model.eval()


    with Image.open(
        image_path
    ) as opened_image:


        image = opened_image.convert(
            "RGB"
        )


        messages = build_prompt_messages(
            image
        )


        inputs = processor.apply_chat_template(

            messages,

            add_generation_prompt=True,

            tokenize=True,

            return_dict=True,

            return_tensors="pt",
        )


    input_length = int(

        inputs[
            "input_ids"
        ].shape[
            1
        ]
    )


    inputs = move_batch_to_device(
        inputs
    )


    with autocast_context():


        generated = model.generate(

            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=False,

            num_beams=1,

            use_cache=True,
        )


    generated_tokens = generated[
        0,
        input_length:
    ]


    decoded = processor.decode(

        generated_tokens,

        skip_special_tokens=True,
    ).strip()


    prediction = normalize_prediction_text(
        decoded
    )


    del inputs
    del generated
    del generated_tokens


    return (
        decoded,
        prediction,
    )


# ==================================================================================================
# 52. METRICS
#
# INVALID MODEL OUTPUT IS COUNTED AS WRONG
# ==================================================================================================

def calculate_binary_metrics(
    true_labels,
    predictions,
):

    metric_predictions = []

    invalid_count = 0


    for truth, prediction in zip(
        true_labels,
        predictions,
    ):


        if prediction is None:


            invalid_count += 1


            metric_predictions.append(

                1
                -
                int(
                    truth
                )
            )


        else:


            metric_predictions.append(
                int(
                    prediction
                )
            )


    y_true = np.asarray(
        true_labels,
        dtype=int,
    )


    y_pred = np.asarray(
        metric_predictions,
        dtype=int,
    )


    cm = confusion_matrix(

        y_true,

        y_pred,

        labels=[
            0,
            1,
        ],
    )


    tn, fp, fn, tp = cm.ravel()


    sensitivity = (

        tp

        /

        (
            tp
            +
            fn
        )

        if
        (
            tp
            +
            fn
        ) > 0

        else

        0.0
    )


    specificity = (

        tn

        /

        (
            tn
            +
            fp
        )

        if
        (
            tn
            +
            fp
        ) > 0

        else

        0.0
    )


    return (

        {

            "TN": int(tn),

            "FP": int(fp),

            "FN": int(fn),

            "TP": int(tp),

            "Sensitivity": float(
                sensitivity
            ),

            "Specificity": float(
                specificity
            ),

            "Balanced_Accuracy": float(
                balanced_accuracy_score(
                    y_true,
                    y_pred,
                )
            ),

            "Accuracy": float(
                accuracy_score(
                    y_true,
                    y_pred,
                )
            ),

            "Precision": float(
                precision_score(
                    y_true,
                    y_pred,
                    zero_division=0,
                )
            ),

            "F1": float(
                f1_score(
                    y_true,
                    y_pred,
                    zero_division=0,
                )
            ),

            "Invalid_Predictions": int(
                invalid_count
            ),
        },

        y_pred,
    )


# ==================================================================================================
# 53. VALIDATION GENERATION
# ==================================================================================================

@torch.no_grad()
def evaluate_validation_generation(
    epoch_number,
):

    model.eval()


    true_labels = []

    predictions = []

    rows = []


    progress = tqdm(

        range(
            len(
                validation_df
            )
        ),

        desc=f"Validation predictions epoch {epoch_number}",
    )


    for index in progress:


        row = validation_df.iloc[
            index
        ]


        image_path = str(
            row[
                "_image_path"
            ]
        )


        # Model sees image and prompts only.
        raw_output, prediction = predict_single_image(
            image_path
        )


        # Ground truth is accessed only after prediction.
        true_label = int(
            row[
                "_label"
            ]
        )


        true_labels.append(
            true_label
        )


        predictions.append(
            prediction
        )


        rows.append(

            {

                "index": index,

                "image_path": image_path,

                "true_label": true_label,

                "true_class": (
                    "Seizure"
                    if
                    true_label == 1
                    else
                    "Non-Seizure"
                ),

                "raw_model_output": raw_output,

                "normalized_prediction": (
                    prediction
                    if
                    prediction is not None
                    else
                    "INVALID"
                ),

                "predicted_class": (
                    "Seizure"
                    if
                    prediction == 1
                    else
                    "Non-Seizure"
                    if
                    prediction == 0
                    else
                    "INVALID"
                ),

            }
        )


    metrics, metric_predictions = calculate_binary_metrics(
        true_labels,
        predictions,
    )


    result_df = pd.DataFrame(
        rows
    )


    result_df[
        "metric_prediction"
    ] = metric_predictions


    result_df[
        "correct"
    ] = (

        result_df[
            "true_label"
        ].to_numpy()

        ==

        metric_predictions
    )


    validation_output_csv = os.path.join(

        OUTPUT_ROOT,

        f"Patient_24_A4D4D3D2_validation_predictions_epoch_{epoch_number}.csv",
    )


    result_df.to_csv(
        validation_output_csv,
        index=False,
    )


    return metrics


# ==================================================================================================
# 54. SAVE INITIAL EXPERIMENT CONFIG
# ==================================================================================================

experiment_config = {

    "Patient": 24,

    "Representation":
        "A4D4D3D2 composite wavelet-coefficient variance topomaps",

    "Model": MODEL_ID,

    "Fine_Tuning_Method": "Standard LoRA",

    "Quantization": "None",

    "Random_Seed": RANDOM_SEED,

    "System_Prompt": SYSTEM_PROMPT,

    "User_Prompt": USER_PROMPT,

    "Training_Metadata": TRAINING_METADATA_CSV,

    "Testing_Metadata": TESTING_METADATA_CSV,

    "Training_Examples": len(
        actual_train_df
    ),

    "Validation_Examples": len(
        validation_df
    ),

    "Testing_Examples": len(
        testing_df
    ),

    "LoRA_R": LORA_R,

    "LoRA_Alpha": LORA_ALPHA,

    "LoRA_Dropout": LORA_DROPOUT,

    "LoRA_Target_Module_Count": len(
        LORA_TARGET_MODULES
    ),

    "Trainable_Parameters": trainable_parameter_count,

    "Total_Parameters": total_parameter_count,

    "Trainable_Percentage": trainable_percentage,

    "Epochs": NUM_EPOCHS,

    "Early_Stopping_Patience":
        EARLY_STOPPING_PATIENCE,

    "Learning_Rate": LEARNING_RATE,

    "Weight_Decay": WEIGHT_DECAY,

    "Physical_Batch_Size":
        TRAIN_BATCH_SIZE,

    "Gradient_Accumulation":
        GRADIENT_ACCUMULATION_STEPS,

    "Effective_Batch_Size": (
        TRAIN_BATCH_SIZE
        *
        GRADIENT_ACCUMULATION_STEPS
    ),

    "Warmup_Ratio":
        WARMUP_RATIO,

    "Max_Grad_Norm":
        MAX_GRAD_NORM,

    "Precision":
        PRECISION_NAME,

    "GPU":
        GPU_NAME,

    "GPU_Memory_GB":
        GPU_MEMORY_GB,

    "PyTorch":
        torch.__version__,

    "Transformers":
        transformers.__version__,

    "PEFT":
        peft.__version__,

    "Leakage_Check_Column":
        LEAKAGE_ID_COLUMN,
}


with open(
    EXPERIMENT_CONFIG_JSON,
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        experiment_config,
        file,
        indent=4,
    )


# ==================================================================================================
# 55. CHECKPOINT ROOT
# ==================================================================================================

print(
    "\nOneDrive-safe checkpoint root:"
)

print(
    BEST_ADAPTER_ROOT
)


# ==================================================================================================
# 56. TRAINING VARIABLES
# ==================================================================================================

training_history = []


best_validation_balanced_accuracy = -float(
    "inf"
)


best_validation_loss = float(
    "inf"
)


best_epoch = None


epochs_without_improvement = 0


stopped_early = False


early_stop_epoch = None


optimizer.zero_grad(
    set_to_none=True
)


gc.collect()
torch.cuda.empty_cache()


torch.cuda.reset_peak_memory_stats(
    DEVICE
)


training_start_time = time.time()


# ==================================================================================================
# 57. START TRAINING
# ==================================================================================================

print("\n" + "=" * 100)
print("STARTING PATIENT 24 A4D4D3D2 MEDGEMMA-4B LoRA TRAINING")
print("=" * 100)


for epoch in range(
    1,
    NUM_EPOCHS + 1,
):


    print("\n" + "=" * 100)

    print(
        f"EPOCH {epoch}/{NUM_EPOCHS}"
    )

    print("=" * 100)


    model.train()


    epoch_losses = []


    epoch_start_time = time.time()


    accumulation_counter = 0


    progress = tqdm(

        enumerate(
            train_loader,
            start=1,
        ),

        total=len(
            train_loader
        ),

        desc=f"Training epoch {epoch}",
    )


    for batch_number, batch in progress:


        try:


            batch = move_batch_to_device(
                batch
            )


            # --------------------------------------------------------------------------------------
            # FORWARD
            # --------------------------------------------------------------------------------------

            with autocast_context():


                outputs = model(
                    **batch
                )


                raw_loss = outputs.loss


                scaled_loss = (

                    raw_loss

                    /

                    GRADIENT_ACCUMULATION_STEPS
                )


            if not torch.isfinite(
                raw_loss
            ):

                raise RuntimeError(
                    f"\nNon-finite training loss detected: "
                    f"{raw_loss.item()}"
                )


            # --------------------------------------------------------------------------------------
            # BACKWARD
            # --------------------------------------------------------------------------------------

            if grad_scaler is not None:


                grad_scaler.scale(
                    scaled_loss
                ).backward()


            else:


                scaled_loss.backward()


            accumulation_counter += 1


            current_loss = float(

                raw_loss
                .detach()
                .float()
                .cpu()
            )


            epoch_losses.append(
                current_loss
            )


            # --------------------------------------------------------------------------------------
            # OPTIMIZER UPDATE
            # --------------------------------------------------------------------------------------

            should_update = (

                accumulation_counter
                >=
                GRADIENT_ACCUMULATION_STEPS

                or

                batch_number
                ==
                len(
                    train_loader
                )
            )


            if should_update:


                if grad_scaler is not None:


                    grad_scaler.unscale_(
                        optimizer
                    )


                torch.nn.utils.clip_grad_norm_(

                    trainable_parameters,

                    MAX_GRAD_NORM,
                )


                if grad_scaler is not None:


                    grad_scaler.step(
                        optimizer
                    )


                    grad_scaler.update()


                else:


                    optimizer.step()


                scheduler.step()


                optimizer.zero_grad(
                    set_to_none=True
                )


                accumulation_counter = 0


            current_lr = scheduler.get_last_lr()[0]


            gpu_allocated = (

                torch.cuda.memory_allocated()

                /

                1024 ** 3
            )


            progress.set_postfix(

                loss=f"{current_loss:.4f}",

                lr=f"{current_lr:.2e}",

                gpu=f"{gpu_allocated:.1f}GB",
            )


            del batch
            del outputs
            del raw_loss
            del scaled_loss


        except torch.cuda.OutOfMemoryError as oom_error:


            print("\n" + "=" * 100)
            print("CUDA OUT OF MEMORY")
            print("=" * 100)


            print(
                f"\nEpoch       : {epoch}"
            )


            print(
                f"Training row: {batch_number}"
            )


            print(
                f"Allocated   : "
                f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
            )


            print(
                f"Reserved    : "
                f"{torch.cuda.memory_reserved() / 1024**3:.2f} GB"
            )


            torch.cuda.empty_cache()


            raise RuntimeError(
                "\nCUDA ran out of memory during standard LoRA training.\n"
                "Do not switch to 4-bit inside this experiment, because "
                "that would make it QLoRA."
            ) from oom_error


    mean_training_loss = float(
        np.mean(
            epoch_losses
        )
    )


    # ----------------------------------------------------------------------------------------------
    # VALIDATION LOSS
    # ----------------------------------------------------------------------------------------------

    print(
        "\nCalculating validation loss..."
    )


    mean_validation_loss = calculate_validation_loss()


    # ----------------------------------------------------------------------------------------------
    # VALIDATION CLASSIFICATION
    # ----------------------------------------------------------------------------------------------

    print(
        "\nRunning validation classification..."
    )


    validation_metrics = evaluate_validation_generation(
        epoch
    )


    validation_ba = validation_metrics[
        "Balanced_Accuracy"
    ]


    epoch_minutes = (

        time.time()

        -

        epoch_start_time

    ) / 60.0


    # ----------------------------------------------------------------------------------------------
    # PRINT EPOCH RESULT
    # ----------------------------------------------------------------------------------------------

    print("\n" + "-" * 100)

    print(
        f"EPOCH {epoch} RESULTS"
    )

    print("-" * 100)


    print(
        f"Training loss          : {mean_training_loss:.6f}"
    )

    print(
        f"Validation loss        : {mean_validation_loss:.6f}"
    )

    print(
        f"TN                     : {validation_metrics['TN']}"
    )

    print(
        f"FP                     : {validation_metrics['FP']}"
    )

    print(
        f"FN                     : {validation_metrics['FN']}"
    )

    print(
        f"TP                     : {validation_metrics['TP']}"
    )

    print(
        f"Sensitivity            : "
        f"{validation_metrics['Sensitivity']:.4f}"
    )

    print(
        f"Specificity            : "
        f"{validation_metrics['Specificity']:.4f}"
    )

    print(
        f"Balanced Accuracy      : "
        f"{validation_metrics['Balanced_Accuracy']:.4f}"
    )

    print(
        f"Accuracy               : "
        f"{validation_metrics['Accuracy']:.4f}"
    )

    print(
        f"Precision              : "
        f"{validation_metrics['Precision']:.4f}"
    )

    print(
        f"F1                     : "
        f"{validation_metrics['F1']:.4f}"
    )

    print(
        f"Invalid outputs        : "
        f"{validation_metrics['Invalid_Predictions']}"
    )

    print(
        f"Epoch duration         : "
        f"{epoch_minutes:.2f} minutes"
    )


    # ----------------------------------------------------------------------------------------------
    # SAVE TRAINING HISTORY
    # ----------------------------------------------------------------------------------------------

    training_history.append(

        {

            "epoch":
                epoch,

            "training_loss":
                mean_training_loss,

            "validation_loss":
                mean_validation_loss,

            "validation_TN":
                validation_metrics["TN"],

            "validation_FP":
                validation_metrics["FP"],

            "validation_FN":
                validation_metrics["FN"],

            "validation_TP":
                validation_metrics["TP"],

            "validation_sensitivity":
                validation_metrics["Sensitivity"],

            "validation_specificity":
                validation_metrics["Specificity"],

            "validation_balanced_accuracy":
                validation_metrics["Balanced_Accuracy"],

            "validation_accuracy":
                validation_metrics["Accuracy"],

            "validation_precision":
                validation_metrics["Precision"],

            "validation_f1":
                validation_metrics["F1"],

            "invalid_predictions":
                validation_metrics["Invalid_Predictions"],

            "epoch_minutes":
                epoch_minutes,
        }

    )


    pd.DataFrame(
        training_history
    ).to_csv(
        TRAINING_HISTORY_CSV,
        index=False,
    )


    # ==============================================================================================
    # BEST CHECKPOINT
    #
    # PRIMARY:
    #     highest validation Balanced Accuracy
    #
    # TIE BREAKER:
    #     lower validation loss
    # ==============================================================================================

    better_checkpoint = False


    if (
        validation_ba
        >
        best_validation_balanced_accuracy
    ):


        better_checkpoint = True


    elif (

        math.isclose(

            validation_ba,

            best_validation_balanced_accuracy,

            rel_tol=0.0,

            abs_tol=1e-12,
        )

        and

        mean_validation_loss
        <
        best_validation_loss
    ):


        better_checkpoint = True


    if better_checkpoint:


        best_validation_balanced_accuracy = validation_ba


        best_validation_loss = mean_validation_loss


        best_epoch = epoch


        # ------------------------------------------------------------------------------------------
        # OneDrive-safe best checkpoint
        # ------------------------------------------------------------------------------------------

        BEST_ADAPTER_DIR = os.path.join(

            BEST_ADAPTER_ROOT,

            f"best_epoch_{epoch:03d}",
        )


        if os.path.exists(
            BEST_ADAPTER_DIR
        ):


            BEST_ADAPTER_DIR = os.path.join(

                BEST_ADAPTER_ROOT,

                f"best_epoch_{epoch:03d}_{time.strftime('%y%m%d_%H%M%S')}",
            )


        os.makedirs(

            BEST_ADAPTER_DIR,

            exist_ok=False,
        )


        # Saves only LoRA adapter weights.
        model.save_pretrained(

            BEST_ADAPTER_DIR,

            safe_serialization=True,
        )


        processor.save_pretrained(
            BEST_ADAPTER_DIR
        )


        print("\n" + "*" * 100)
        print("NEW BEST LoRA CHECKPOINT SAVED")
        print("*" * 100)


        print(
            f"\nBest epoch      : {best_epoch}"
        )


        print(
            f"Validation BA   : "
            f"{best_validation_balanced_accuracy:.4f}"
        )


        print(
            f"Validation loss : "
            f"{best_validation_loss:.6f}"
        )


        print(
            "\nAdapter saved to:"
        )


        print(
            BEST_ADAPTER_DIR
        )


        # Reset early stopping counter.
        epochs_without_improvement = 0


        print(
            f"Early stopping    : 0/{EARLY_STOPPING_PATIENCE}"
        )


    else:


        epochs_without_improvement += 1


        print("\n" + "-" * 100)


        print(

            f"NO NEW BEST CHECKPOINT | "

            f"Early stopping patience: "

            f"{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
        )


        print("-" * 100)


    gc.collect()

    torch.cuda.empty_cache()


    # ==============================================================================================
    # EARLY STOPPING
    # ==============================================================================================

    if (
        epochs_without_improvement
        >=
        EARLY_STOPPING_PATIENCE
    ):


        stopped_early = True


        early_stop_epoch = epoch


        print("\n" + "=" * 100)
        print("EARLY STOPPING TRIGGERED")
        print("=" * 100)


        print(
            f"\nNo new best checkpoint for "
            f"{EARLY_STOPPING_PATIENCE} consecutive epochs."
        )


        print(
            f"Stopping at epoch     : {early_stop_epoch}"
        )


        print(
            f"Best epoch            : {best_epoch}"
        )


        print(
            f"Best validation BA    : "
            f"{best_validation_balanced_accuracy:.4f}"
        )


        print(
            f"Best validation loss  : "
            f"{best_validation_loss:.6f}"
        )


        break


# ==================================================================================================
# 58. TRAINING COMPLETE
# ==================================================================================================

total_training_minutes = (

    time.time()

    -

    training_start_time

) / 60.0


peak_training_gpu_allocated_gb = (

    torch.cuda.max_memory_allocated(
        DEVICE
    )

    /

    1024 ** 3
)


peak_training_gpu_reserved_gb = (

    torch.cuda.max_memory_reserved(
        DEVICE
    )

    /

    1024 ** 3
)


print("\n" + "=" * 100)
print("LoRA TRAINING COMPLETE")
print("=" * 100)


print(
    f"\nBest epoch           : {best_epoch}"
)


print(
    f"Best validation BA   : "
    f"{best_validation_balanced_accuracy:.4f}"
)


print(
    f"Best validation loss : "
    f"{best_validation_loss:.6f}"
)


print(
    f"Early stop patience  : "
    f"{EARLY_STOPPING_PATIENCE}"
)


print(
    f"Stopped early        : "
    f"{stopped_early}"
)


if stopped_early:

    print(
        f"Stopped at epoch     : "
        f"{early_stop_epoch}"
    )


print(
    f"Training time        : "
    f"{total_training_minutes:.2f} minutes"
)


print(
    f"Peak GPU allocated   : "
    f"{peak_training_gpu_allocated_gb:.2f} GB"
)


print(
    f"Peak GPU reserved    : "
    f"{peak_training_gpu_reserved_gb:.2f} GB"
)


if best_epoch is None:

    raise RuntimeError(
        "\nNo valid LoRA checkpoint was created."
    )


# ==================================================================================================
# 59. FREE TRAINING MODEL
# ==================================================================================================

print("\n" + "=" * 100)
print("FREEING TRAINING MODEL")
print("=" * 100)


del model
del base_model
del optimizer
del scheduler
del trainable_parameters


gc.collect()

torch.cuda.empty_cache()


print(
    f"\nGPU allocated after cleanup: "
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)


# ==================================================================================================
# 60. LOAD FRESH MEDGEMMA FOR FINAL TEST
# ==================================================================================================

print("\n" + "=" * 100)
print("LOADING FRESH MEDGEMMA + BEST LoRA ADAPTER")
print("=" * 100)


base_model_for_test = load_medgemma_base()


if hasattr(
    base_model_for_test.config,
    "use_cache",
):

    base_model_for_test.config.use_cache = True


# ==================================================================================================
# 61. LOAD BEST LoRA ADAPTER
# ==================================================================================================

if BEST_ADAPTER_DIR is None:

    raise RuntimeError(
        "No best LoRA adapter was saved during training."
    )


if not os.path.isdir(
    BEST_ADAPTER_DIR
):

    raise FileNotFoundError(
        f"Best LoRA adapter directory does not exist:\n"
        f"{BEST_ADAPTER_DIR}"
    )


print(
    f"\nLoading best adapter from:\n"
    f"{BEST_ADAPTER_DIR}"
)


model = PeftModel.from_pretrained(

    base_model_for_test,

    BEST_ADAPTER_DIR,

    is_trainable=False,
)


model = model.to(
    DEVICE
)


model.eval()


print(
    "\nSUCCESS: Best LoRA adapter loaded."
)


print(
    f"Using checkpoint from epoch {best_epoch}."
)


# ==================================================================================================
# 62. FINAL BLINDED TEST
#
# Ground truth NEVER enters the messages sent to MedGemma.
# ==================================================================================================

print("\n" + "=" * 100)

print(
    f"STARTING FINAL {len(testing_df)}-IMAGE TEST"
)

print("=" * 100)


test_rows = []

test_true_labels = []

test_predictions = []


torch.cuda.reset_peak_memory_stats(
    DEVICE
)


test_start_time = time.time()


progress = tqdm(

    range(
        len(
            testing_df
        )
    ),

    desc="Final blinded test",
)


for index in progress:


    row = testing_df.iloc[
        index
    ]


    image_path = str(
        row[
            "_image_path"
        ]
    )


    # ----------------------------------------------------------------------------------------------
    # 1. MODEL PREDICTION FIRST
    # ----------------------------------------------------------------------------------------------

    raw_output, prediction = predict_single_image(
        image_path
    )


    # ----------------------------------------------------------------------------------------------
    # 2. GROUND TRUTH ACCESSED AFTER PREDICTION
    # ----------------------------------------------------------------------------------------------

    true_label = int(
        row[
            "_label"
        ]
    )


    test_true_labels.append(
        true_label
    )


    test_predictions.append(
        prediction
    )


    test_rows.append(

        {

            "test_index":
                index,

            "image_path":
                image_path,

            "true_label":
                true_label,

            "true_class":
                (
                    "Seizure"
                    if
                    true_label == 1
                    else
                    "Non-Seizure"
                ),

            "raw_model_output":
                raw_output,

            "normalized_prediction":
                (
                    prediction
                    if
                    prediction is not None
                    else
                    "INVALID"
                ),

            "predicted_class":
                (
                    "Seizure"
                    if
                    prediction == 1
                    else
                    "Non-Seizure"
                    if
                    prediction == 0
                    else
                    "INVALID"
                ),

        }

    )


    # ----------------------------------------------------------------------------------------------
    # PERIODIC BACKUP
    # ----------------------------------------------------------------------------------------------

    if (

        (index + 1) % 25 == 0

        or

        (index + 1)
        ==
        len(
            testing_df
        )

    ):


        temporary_prediction_df = pd.DataFrame(
            test_rows
        )


        temporary_prediction_df.to_csv(

            FINAL_PREDICTIONS_CSV,

            index=False,
        )


# ==================================================================================================
# 63. FINAL METRICS
# ==================================================================================================

final_metrics, final_metric_predictions = calculate_binary_metrics(

    test_true_labels,

    test_predictions,
)


# ==================================================================================================
# 64. FINAL PREDICTION TABLE
# ==================================================================================================

final_prediction_df = pd.DataFrame(
    test_rows
)


final_prediction_df[
    "metric_prediction"
] = final_metric_predictions


final_prediction_df[
    "metric_predicted_class"
] = [

    "Seizure"

    if
    prediction == 1

    else

    "Non-Seizure"

    for prediction
    in final_metric_predictions
]


final_prediction_df[
    "correct"
] = (

    final_prediction_df[
        "true_label"
    ].to_numpy()

    ==

    final_metric_predictions
)


# ==================================================================================================
# 65. PRESERVE ALL ORIGINAL TEST METADATA
# ==================================================================================================

original_testing_columns = testing_df.drop(

    columns=[
        "_image_path",
        "_label",
    ],

    errors="ignore",

).copy()


original_testing_columns = original_testing_columns.add_prefix(
    "metadata_"
)


final_prediction_df = pd.concat(

    [

        final_prediction_df.reset_index(
            drop=True
        ),

        original_testing_columns.reset_index(
            drop=True
        ),

    ],

    axis=1,
)


# ==================================================================================================
# 66. SAVE FINAL PREDICTIONS
# ==================================================================================================

final_prediction_df.to_csv(

    FINAL_PREDICTIONS_CSV,

    index=False,
)


# ==================================================================================================
# 67. FINAL TEST TIME / MEMORY
# ==================================================================================================

final_test_minutes = (

    time.time()

    -

    test_start_time

) / 60.0


peak_test_gpu_allocated_gb = (

    torch.cuda.max_memory_allocated(
        DEVICE
    )

    /

    1024 ** 3
)


peak_test_gpu_reserved_gb = (

    torch.cuda.max_memory_reserved(
        DEVICE
    )

    /

    1024 ** 3
)


# ==================================================================================================
# 68. CONFUSION MATRIX CSV
# ==================================================================================================

confusion_matrix_df = pd.DataFrame(

    [

        [

            final_metrics[
                "TN"
            ],

            final_metrics[
                "FP"
            ],

        ],

        [

            final_metrics[
                "FN"
            ],

            final_metrics[
                "TP"
            ],

        ],

    ],

    index=[
        "Actual_Non_Seizure",
        "Actual_Seizure",
    ],

    columns=[
        "Predicted_Non_Seizure",
        "Predicted_Seizure",
    ],
)


confusion_matrix_df.to_csv(
    CONFUSION_MATRIX_CSV
)


# ==================================================================================================
# 69. ADD EXPERIMENT INFORMATION TO FINAL METRICS
# ==================================================================================================

final_metrics[
    "Patient"
] = 24


final_metrics[
    "Representation"
] = "A4D4D3D2 composite wavelet-coefficient variance topomaps"


final_metrics[
    "Base_Model"
] = MODEL_ID


final_metrics[
    "Method"
] = "Standard LoRA"


final_metrics[
    "Quantization"
] = "None"


final_metrics[
    "Precision_Dtype"
] = PRECISION_NAME


final_metrics[
    "Best_Epoch"
] = int(
    best_epoch
)


final_metrics[
    "Best_Validation_Balanced_Accuracy"
] = float(
    best_validation_balanced_accuracy
)


final_metrics[
    "Best_Validation_Loss"
] = float(
    best_validation_loss
)


final_metrics[
    "LoRA_Rank"
] = int(
    LORA_R
)


final_metrics[
    "LoRA_Alpha"
] = int(
    LORA_ALPHA
)


final_metrics[
    "LoRA_Dropout"
] = float(
    LORA_DROPOUT
)


final_metrics[
    "Trainable_Parameters"
] = int(
    trainable_parameter_count
)


final_metrics[
    "Total_Parameters"
] = int(
    total_parameter_count
)


final_metrics[
    "Trainable_Percentage"
] = float(
    trainable_percentage
)


final_metrics[
    "Training_Time_Minutes"
] = float(
    total_training_minutes
)


final_metrics[
    "Final_Test_Time_Minutes"
] = float(
    final_test_minutes
)


final_metrics[
    "Peak_Training_GPU_Allocated_GB"
] = float(
    peak_training_gpu_allocated_gb
)


final_metrics[
    "Peak_Training_GPU_Reserved_GB"
] = float(
    peak_training_gpu_reserved_gb
)


final_metrics[
    "Peak_Test_GPU_Allocated_GB"
] = float(
    peak_test_gpu_allocated_gb
)


final_metrics[
    "Peak_Test_GPU_Reserved_GB"
] = float(
    peak_test_gpu_reserved_gb
)


# ==================================================================================================
# 70. SAVE FINAL METRICS
# ==================================================================================================

with open(
    FINAL_METRICS_JSON,
    "w",
    encoding="utf-8",
) as file:


    json.dump(
        final_metrics,
        file,
        indent=4,
    )


# ==================================================================================================
# 71. UPDATE EXPERIMENT CONFIG
# ==================================================================================================

experiment_config[
    "Best_Epoch"
] = int(
    best_epoch
)


experiment_config[
    "Best_Validation_Balanced_Accuracy"
] = float(
    best_validation_balanced_accuracy
)


experiment_config[
    "Best_Validation_Loss"
] = float(
    best_validation_loss
)


experiment_config[
    "Training_Time_Minutes"
] = float(
    total_training_minutes
)


experiment_config[
    "Final_Test_Time_Minutes"
] = float(
    final_test_minutes
)


experiment_config[
    "Peak_Training_GPU_Allocated_GB"
] = float(
    peak_training_gpu_allocated_gb
)


experiment_config[
    "Peak_Training_GPU_Reserved_GB"
] = float(
    peak_training_gpu_reserved_gb
)


experiment_config[
    "Peak_Test_GPU_Allocated_GB"
] = float(
    peak_test_gpu_allocated_gb
)


experiment_config[
    "Peak_Test_GPU_Reserved_GB"
] = float(
    peak_test_gpu_reserved_gb
)


with open(
    EXPERIMENT_CONFIG_JSON,
    "w",
    encoding="utf-8",
) as file:


    json.dump(
        experiment_config,
        file,
        indent=4,
    )


# ==================================================================================================
# 72. FINAL REPORT
# ==================================================================================================

print("\n" + "=" * 100)

print(
    "PATIENT 24 A4D4D3D2 - MEDGEMMA-4B LoRA FINAL RESULTS"
)

print("=" * 100)


print("\nEXPERIMENT")


print(
    f"Base model        : {MODEL_ID}"
)


print(
    "Method            : STANDARD LoRA"
)


print(
    "Quantization      : NONE"
)


print(
    f"Precision         : {PRECISION_NAME}"
)


print(
    f"Best epoch        : {best_epoch}"
)


print("\n" + "-" * 100)
print("LoRA CONFIGURATION")
print("-" * 100)


print(
    f"\nRank r            : {LORA_R}"
)


print(
    f"Alpha             : {LORA_ALPHA}"
)


print(
    f"Dropout           : {LORA_DROPOUT}"
)


print(
    "Target            : language q_proj, k_proj, v_proj, o_proj"
)


print(
    f"Target modules    : {len(LORA_TARGET_MODULES)}"
)


print("\n" + "-" * 100)
print("DATASET")
print("-" * 100)


print(
    "\nActual training"
)


print(
    f"Seizure     : {actual_train_seizure_count}"
)


print(
    f"Non-seizure : {actual_train_non_seizure_count}"
)


print(
    f"Total        : {len(actual_train_df)}"
)


print(
    "\nValidation"
)


print(
    f"Seizure     : {validation_seizure_count}"
)


print(
    f"Non-seizure : {validation_non_seizure_count}"
)


print(
    f"Total        : {len(validation_df)}"
)


print(
    "\nFinal testing"
)


print(
    f"Seizure     : {test_seizure_count}"
)


print(
    f"Non-seizure : {test_non_seizure_count}"
)


print(
    f"Total        : {len(testing_df)}"
)


print("\n" + "-" * 100)
print("CONFUSION MATRIX")
print("-" * 100)


print(
    f"\nTN = {final_metrics['TN']}"
)


print(
    f"FP = {final_metrics['FP']}"
)


print(
    f"FN = {final_metrics['FN']}"
)


print(
    f"TP = {final_metrics['TP']}"
)


print("\n" + "-" * 100)
print("CLASSIFICATION METRICS")
print("-" * 100)


print(
    f"\nSensitivity       : "
    f"{final_metrics['Sensitivity']:.4f}"
)


print(
    f"Specificity       : "
    f"{final_metrics['Specificity']:.4f}"
)


print(
    f"Balanced Accuracy : "
    f"{final_metrics['Balanced_Accuracy']:.4f}"
)


print(
    f"Accuracy          : "
    f"{final_metrics['Accuracy']:.4f}"
)


print(
    f"Precision         : "
    f"{final_metrics['Precision']:.4f}"
)


print(
    f"F1 Score          : "
    f"{final_metrics['F1']:.4f}"
)


print(
    f"Invalid outputs   : "
    f"{final_metrics['Invalid_Predictions']}"
)


print("\n" + "-" * 100)
print("PARAMETER EFFICIENCY")
print("-" * 100)


print(
    f"\nTrainable parameters : "
    f"{trainable_parameter_count:,}"
)


print(
    f"Total parameters     : "
    f"{total_parameter_count:,}"
)


print(
    f"Trainable percentage : "
    f"{trainable_percentage:.6f}%"
)


print("\n" + "-" * 100)
print("TIME / GPU MEMORY")
print("-" * 100)


print(
    f"\nTraining time           : "
    f"{total_training_minutes:.2f} minutes"
)


print(
    f"Final test time         : "
    f"{final_test_minutes:.2f} minutes"
)


print(
    f"Peak training allocated : "
    f"{peak_training_gpu_allocated_gb:.2f} GB"
)


print(
    f"Peak training reserved  : "
    f"{peak_training_gpu_reserved_gb:.2f} GB"
)


print(
    f"Peak testing allocated  : "
    f"{peak_test_gpu_allocated_gb:.2f} GB"
)


print(
    f"Peak testing reserved   : "
    f"{peak_test_gpu_reserved_gb:.2f} GB"
)


print("\n" + "=" * 100)
print("FILES SAVED")
print("=" * 100)


print(
    "\n1. Best LoRA adapter:"
)


print(
    BEST_ADAPTER_DIR
)


print(
    "\nCheckpoint root containing all saved best epochs:"
)


print(
    BEST_ADAPTER_ROOT
)


print(
    "\n2. Exact training split:"
)


print(
    TRAIN_SPLIT_CSV
)


print(
    "\n3. Exact validation split:"
)


print(
    VALIDATION_SPLIT_CSV
)


print(
    "\n4. Training history:"
)


print(
    TRAINING_HISTORY_CSV
)


print(
    f"\n5. Final {len(testing_df)}-image predictions:"
)


print(
    FINAL_PREDICTIONS_CSV
)


print(
    "\n6. Final metrics:"
)


print(
    FINAL_METRICS_JSON
)


print(
    "\n7. Confusion matrix CSV:"
)


print(
    CONFUSION_MATRIX_CSV
)


print(
    "\n8. Experiment configuration:"
)


print(
    EXPERIMENT_CONFIG_JSON
)


print(
    "\n9. Exact LoRA target modules:"
)


print(
    TARGET_MODULES_TXT
)


print("\n" + "=" * 100)

print(
    "PATIENT 24 A4D4D3D2 MEDGEMMA-4B STANDARD LoRA EXPERIMENT COMPLETE"
)

print("=" * 100)